# Local ColabFold MSA Notebook

- This notebook is adapted from the Colab-oriented MMseqs2 ColabFold workflow in this repository.
- It keeps the key parameters visible in notebook cells, but replaces Google Colab upload and Google Drive steps with local file paths.
- The default workflow focuses on submitting MSA jobs to api.colabfold.com from a local Python environment.

Before running this notebook, install colabfold first

```shell
mamba create -p /opt/ColabFold/.env python=3.11
pip install colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold
```

# Configure ColabFold MSA Submission

Set the local input/output paths and the main ColabFold MSA parameters here.

In [ ]:
# @title Editing submission
from pathlib import Path

WORK_DIR = Path.cwd() / "local_colabfold_jobs"
QUERY_BATCH_NAME = "hPKR_binders_local"
RESULT_DIR_NAME = QUERY_BATCH_NAME
OVERWRITE_RESULTS = False

# Primary input: sequence dict
SEQUENCE_DICT = {
    "binder_1": "MKTAYIAKQRQISFVKSHFSRQLEERLGLIEVQANL",
    "binder_2": "GHHHHHHMEEAKEAELKQAGVDTYLQTKV",
}

# Optional export: write SEQUENCE_DICT to FASTA for compatibility/debugging
EXPORT_SEQUENCE_DICT_TO_FASTA = True
EXPORTED_FASTA_PATH = WORK_DIR / f"{QUERY_BATCH_NAME}.fasta"

# MSA options
MSA_MODE = "mmseqs2_uniref_env"  # options: mmseqs2_uniref_env, mmseqs2_uniref, single_sequence
PAIR_MODE = "unpaired"  # options: unpaired_paired, paired, unpaired
PAIRING_STRATEGY = "greedy"  # options: greedy, complete

# Template options
TEMPLATE_MODE = "none"  # options: none, pdb100, custom
CUSTOM_TEMPLATE_DIR = None  # example: WORK_DIR / "templates"

# Execution options
SHOW_MSA_PLOTS = True
USER_AGENT = "local-colabfold-notebook"
NUM_MODELS = 0
STOP_AT_SCORE = 100.0
KEEP_EXISTING_RESULTS = False

WORK_DIR.mkdir(parents=True, exist_ok=True)
print(f"Working directory: {WORK_DIR.resolve()}")
print(f"Configured sequence entries: {len(SEQUENCE_DICT)}")
print(f"Export FASTA path: {Path(EXPORTED_FASTA_PATH).resolve()}")

In [ ]:
# @title Prepare local inputs
import shutil

jobname = QUERY_BATCH_NAME.strip().replace(" ", "_")
result_dir = WORK_DIR / RESULT_DIR_NAME

if result_dir.exists():
    if OVERWRITE_RESULTS:
        shutil.rmtree(result_dir)
    else:
        raise FileExistsError(f"Result directory already exists: {result_dir}")

result_dir.mkdir(parents=True, exist_ok=False)

if not isinstance(SEQUENCE_DICT, dict) or not SEQUENCE_DICT:
    raise ValueError("SEQUENCE_DICT must be a non-empty dict of {job_id: sequence}.")

normalized_sequence_dict = {}
for seq_id, sequence in SEQUENCE_DICT.items():
    normalized_id = str(seq_id).strip()
    normalized_sequence = "".join(str(sequence).split()).upper()
    if not normalized_id:
        raise ValueError("Each sequence id in SEQUENCE_DICT must be non-empty.")
    if not normalized_sequence:
        raise ValueError(f"Sequence for '{normalized_id}' is empty after normalization.")
    normalized_sequence_dict[normalized_id] = normalized_sequence

queries_for_run = [
    (seq_id, sequence, None, None)
    for seq_id, sequence in normalized_sequence_dict.items()
 ]

is_complex = any(":" in sequence for sequence in normalized_sequence_dict.values())

exported_fasta_path = Path(EXPORTED_FASTA_PATH)
if EXPORT_SEQUENCE_DICT_TO_FASTA:
    exported_fasta_path.parent.mkdir(parents=True, exist_ok=True)
    with open(exported_fasta_path, "w", encoding="utf-8") as handle:
        for seq_id, sequence in normalized_sequence_dict.items():
            handle.write(f">{seq_id}\n{sequence}\n")
    print(f"Exported FASTA: {exported_fasta_path.resolve()}")

if TEMPLATE_MODE == "pdb100":
    use_templates = True
    custom_template_path = None
elif TEMPLATE_MODE == "custom":
    if CUSTOM_TEMPLATE_DIR is None:
        raise ValueError("CUSTOM_TEMPLATE_DIR must be set when TEMPLATE_MODE is 'custom'.")
    custom_template_path = Path(CUSTOM_TEMPLATE_DIR)
    if not custom_template_path.exists():
        raise FileNotFoundError(f"Custom template directory does not exist: {custom_template_path}")
    use_templates = True
else:
    custom_template_path = None
    use_templates = False

print(f"Prepared {len(queries_for_run)} in-memory queries")
print(f"Result directory: {result_dir.resolve()}")
print(f"Detected complex mode: {is_complex}")

In [ ]:
# @title Submitting queries to https://api.colabfold.com
import warnings
from pathlib import Path

warnings.simplefilter(action="ignore", category=FutureWarning)
from Bio import BiopythonDeprecationWarning
warnings.simplefilter(action="ignore", category=BiopythonDeprecationWarning)

from colabfold.batch import run, set_model_type
from colabfold.plot import plot_msa_v2
import matplotlib.pyplot as plt

def input_features_callback(input_features):
    if SHOW_MSA_PLOTS:
        plot_msa_v2(input_features)
        plt.show()
        plt.close()

model_type = set_model_type(is_complex, "auto")
use_cluster_profile = "multimer" not in model_type

results = run(
    queries=queries_for_run,
    result_dir=result_dir,
    use_templates=use_templates,
    custom_template_path=custom_template_path,
    msa_mode=MSA_MODE,
    model_type=model_type,
    num_models=NUM_MODELS,
    is_complex=is_complex,
    data_dir=Path("."),
    keep_existing_results=KEEP_EXISTING_RESULTS,
    rank_by="auto",
    pair_mode=PAIR_MODE,
    pairing_strategy=PAIRING_STRATEGY,
    stop_at_score=float(STOP_AT_SCORE),
    prediction_callback=None,
    zip_results=False,
    use_cluster_profile=use_cluster_profile,
    input_features_callback=input_features_callback if SHOW_MSA_PLOTS else None,
    user_agent=USER_AGENT,
)

print("Submission finished. Result files created under:")
print(result_dir.resolve())
results

In [ ]:
# @title Preview generated result files
from pathlib import Path

result_files = [path for path in sorted(result_dir.rglob("*")) if path.is_file()]
if not result_files:
    print("No result files were found. Run the submission cell first.")
else:
    print("Generated files:")
    for path in result_files[:50]:
        print(path.relative_to(result_dir))

    preview_candidates = [path for path in result_files if path.suffix in {".a3m", ".fasta", ".csv", ".json"}]
    if preview_candidates:
        preview_path = preview_candidates[0]
        print(f"\nPreviewing: {preview_path.relative_to(result_dir)}\n")
        with open(preview_path, "r", encoding="utf-8", errors="ignore") as handle:
            for line_number, line in enumerate(handle):
                if line_number >= 40:
                    break
                print(line.rstrip())

# Notes

- This local notebook intentionally keeps `NUM_MODELS = 0` by default, so the workflow stays MSA-focused.
- For a pure local MSA submission workflow, `TEMPLATE_MODE = "none"` is usually the simplest choice.
- If you want the full structure prediction workflow later, you can turn template-related settings back on and add the heavier dependencies separately.